# Libs

In [3]:
import pandas as pd
import numpy as np
import json
import os
import datetime as dt
import pickle
from unidecode import unidecode
from sklearn import metrics
from pytimeparse.timeparse import timeparse
import matplotlib.pyplot as plt
import shap
import time
import warnings
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

warnings.filterwarnings('ignore')

In [4]:
src_path = os.path.abspath(os.getcwd())
root_path = os.path.dirname(src_path)
data_path = os.path.join(root_path, 'dados')
raw_path = os.path.join(data_path, 'raw')
interim_path = os.path.join(data_path, 'interim')
deputados_path = os.path.join(raw_path, 'deputados')
discursos_path = os.path.join(deputados_path, 'discursos')

In [5]:
def flatten_dict(d, parent_key='', sep='_'):
    """
    Flatten a nested dictionary.

    Args:
    - d (dict): The dictionary to flatten.
    - parent_key (str): The base key string.
    - sep (str): The separator between keys.

    Returns:
    - dict: The flattened dictionary.
    """
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

In [6]:
def translate_text(text, max_length=5000):
    text = text.replace('\r', ' ').replace('\n', ' ').replace('\t', ' ').replace('\xa0','')
    if len(text) > max_length:
        translations = []
        for i in range(0, len(text), max_length):
            chunk = text[i:i+max_length]
            translation = translator.translate(chunk, src='pt', dest='en').text
            translations.append(translation)
        return ' '.join(translations)
    else:
        translation_result = translator.translate(text, src='pt', dest='en')
        translated_text = translation_result.text
        if translated_text is None:
            translated_text = ""
        return translated_text

In [7]:
def safe_translate(text, max_retries=3):
    attempt = 0
    while attempt < max_retries:
        try:
            return translate_text(text)
        except Exception as e:
            print(f"Translation error: {e}")
            attempt += 1
            time.sleep(2)  # Wait for 2 seconds before retrying
    return None  # Return None if all attempts fail

In [8]:
def track_progress(future, index):
    if index % 1000 == 0:
        print(f"Completed {index} translations")

In [98]:
def get_sentiment(text, flag):
    if flag == 'blob':
        blob = TextBlob(text)
        return blob.sentiment.polarity, blob.sentiment.subjectivity
    else:        
        sia_score = sia.polarity_scores(text) 
        return sia_score['neg'], sia_score['neu'], sia_score['pos'], sia_score['compound']

# Text Analysis

## Open data

In [9]:
discursos_path

'D:\\Pedro\\Desktop\\Dropbox\\UnB\\Doc\\projects\\api_camara\\dados\\raw\\deputados\\discursos'

In [10]:
# with open(os.path.join(discursos_path,'deputados_discursos_raw_.json'),encoding='ISO-8859-1') as json_file:
#     json_discursos = json.load(json_file)    

In [11]:
# len(json_discursos.keys())

In [12]:
# df_list = []
# keys = [x for x in json_discursos.keys()]
# for i, key in enumerate(keys):
#     values_list =  json_discursos[key]
#     if i%100==0:
#         print(f" Appending {key} - ({i}/{len(keys)}) - size {len(values_list)}")
#     for dict_ in values_list:
#         flattened_dict = flatten_dict(dict_)
#         df_ = pd.DataFrame(flattened_dict, index = [key])
#         df_list.append(df_)

In [13]:
# df_speech = pd.concat(df_list)

In [14]:
# df_speech = df_speech.reset_index().rename(columns={'index':'id_deputado'})

In [15]:
# df_speech.to_csv(os.path.join(discursos_path,'deputados_discursos_raw_.json'),index=False)

In [133]:
dict_tematic

{'Agropecuária': ['agropecuária',
  'agronegócio',
  'agrícola',
  'fazenda',
  'plantação',
  'plantações',
  'animais',
  'animal',
  'agrotóxico',
  'pecuária',
  'bovino',
  'suíno',
  'avicultura',
  'piscicultura'],
 'Cidades e Transportes': ['cidades',
  'transportes',
  'trânsito',
  'mobilidade',
  'veículos',
  'urbanismo',
  'predial',
  'metrópole',
  'metropolitano',
  'território',
  'cargas',
  'acessibilidade',
  'passageiros',
  'motorizado',
  'terminais',
  'metroviário',
  'metrô',
  'ferroviário',
  'ferrovia',
  'hidroviário',
  'hidrovia',
  'ciclovias',
  'aeroportos'],
 'Ciência, Tecnologia e comunicações': ['ciência',
  'tecnologia',
  'televisão',
  'inovação',
  'rádio',
  'cnpq',
  'capes'],
 'Economia & Consumidor': ['economia',
  'econômic',
  'indústria',
  'capital',
  'juros',
  'finança',
  'consumidor',
  'dívida',
  'comércio',
  'imposto',
  'empresário',
  'investidor',
  'macroeconomi',
  'câmbio',
  'empresa',
  'inflação'],
 'Direitos Humanos':

In [16]:
dict_tematic = {
    'Agropecuária': [
        'agro',
        'agrícola',
        'fazenda',
        'plantação',
        'plantações',
        'animais',
        'animal',
        'agrotóxico',
        'pecuária',
        'bovino',
        'suíno',
        'avicultura',
        'piscicultura'
                ],
    'Cidades e Transportes':[
        'cidades',
        'transportes',
        'trânsito',
        'mobilidade',
        'veículos',
        'urbanismo',
        'predial',
        'metrópole',
        'metropolitano',
        'território',
        'cargas',
        'acessibilidade',
        'passageiros',
        'motorizado',
        'terminais',
        'metroviário',
        'metrô',
        'ferroviário',
        'ferrovia',
        'hidroviário',
        'hidrovia',
        'ciclovias',
        'aeroportos'
        ],
    'Ciência, Tecnologia e comunicações':[
        'ciência',
        'tecnologia',
        'televisão',
        'inovação',
        'rádio',
        'cnpq',
        'capes',
        ],
    'Economia & Consumidor':[
        'economia',
        'econômic',
        'indústria',
        'capital',
        'juros',
        'finança',
        'consumidor',
        'dívida',
        'comércio',
        'imposto',
        'empresário',
        'empresa',
        'investidor',
        'macroeconomi',
        'câmbio',
        'empresa',
        'inflação',
        ],
    'Direitos Humanos':[
        'direitos humanos',
        'LGBT',
        'minorias',
        'gay',
        'lésbic'
    ],
    'Educação, cultura e esportes':[
        'educação',
        'cultura',
        'esporte',
        'MEC',
        'aluno',
        'estudante',
        'universi',
        'escola',
        'professor',
        'campus',
        'docente',
        'pedagogia',
        'creche',
        'ensino',
        'FUNDEB',
        'infância',
        'PROUNI',
        # 'institutos federais',
        'mensalidade',
        'charter',
        ],
    'Meio ambiente e energia':[
        'meio ambiente',
        'energia',
        'eólica',
        'elétrica',
        'PETROBRAS',
        'ELETROBRAS',
        'combustíveis fósseis',
        'hidroelétrica',
        'termoelétrica',
        'gás',
        'renovável',
        'renováveis',
        'fotovoltaica',
        'poluição',
        'biomassa',
        'etanol',
        'biocombustível',
        'estufa',
        'metano',
        'particulados',
        'biodiesel',
        ],
    'Política e administração pública':[
        'política',
        'administração pública',
        'estado',
        'município',
        ],
    'Relações exteriores':[
        'relações exteriores',
        'diplomacia',
        'itamaraty',
        'embaixador',
        'política externa',
        'política comercial',
        'comércio internacional',
        # 'ernesto araújo',
        'mercosul',
        'trump',
        'ex-tarifário',
        'exporta',
        'barreira tarifária',
        'importação',
        'importações',
        'união européia',
        'venezuela',
        'alfândega',
        'OCDE',
        'conteúdo local',
        'imigração',
        'conselho de segurança',
        'fronteira',
        'oriente médio',
        'china',
        'consul'
        ],
    'Saúde' :[
        'saúde',
        'SUS',
        'médicos',
        'médico',
        'médica',
        'vacinação',
        'remédio',
        'remédios'
        ],
    'Segurança': [
        'segurança',
        'violência',
        'maioridade penal',
        'presídio',
        'presídios',
        'assassinatos',
        'estupro',
        'maria da penha',
        'feminicídio' 
        ],
    'Trabalho, previdência e assistência':[
        'trabalho',
        'previdência',
        'assistência social',
        ]
}

In [17]:
# with open(os.path.join(interim_path,'tematics-words.json'), 'w') as f:
#     json.dump(dict_tematic, f, ensure_ascii=False, indent=4)

In [18]:
with open(os.path.join(interim_path,'tematics-words.json')) as f:
    dict_tematic = json.load(f)

In [19]:
df_speech = pd.read_csv(os.path.join(discursos_path,'deputados_discursos_raw_.json'))
df_speech = df_speech.dropna(subset = ['transcricao'])
df_speech.dataHoraInicio = pd.to_datetime(df_speech.dataHoraInicio)

In [20]:
df_dep_info = pd.read_csv(os.path.join(interim_path,'dep_info.csv'), sep = ';')

In [21]:
list_of_lists = df_dep_info.nome.str.split(' ').to_list()
stop_words1 = [item.upper() for sublist in list_of_lists for item in sublist]

In [22]:
stop_words2 = ['GILDENEMYR', 'OTACI',
              "Acre","AC","Alagoas","AL","Amapá","AP","Amazonas","AM","Bahia","BA","Ceará","CE","Distrito","DF","Espírito","Santo","ES","Goiás","GO","Maranhão","MA",
              "Mato","Grosso","MT","sul","MS","Minas","Gerais","MG","Pará","PA","Paraíba","PB","paraná","PR","Pernambuco","PE","Piauí","PI","Rio","Janeiro","RJ","Norte","RN",
              "RS","Rondônia","RO","Roraima","RR","Santa","Catarina","SC","São","Paulo","SP","Sergipe","SE","Tocantins","TO",
              "DEM","PP","PDT","PODE","PSB","PT","PRB","CIDADANIA","PSOL","MDB","PCdoB","PSD","PL","PTB","PROS","PSDB","PSC","PSL","SOLIDARIEDADE","NOVO","AVANTE","PATRIOTA","PV","S.PART.","REDE","PMN","PHS",
              'líder','deputada','sem','bloco','deputado','dr','-',".",",","...","(",")",'sra','','sras','srs','sr.','sr','sra.','sem','sra','presidente','deputados','deputadas','pela',
              'ordem','revisão','orador','discurso','de','a','o','que','e','do','da','em','um','para','e','com','uma','os','no','se','na','por','as','dos','como','foi','ao','ele','das','tem', 
              'a','seu','sua','ou','ser','quando','há','é','nós','nos','ja','esta','eu','tambem','so','pelo','pela','ate','isso','ela','entre','era','depois','aos','ter','seus',
              'quem','nas','me','esse','eles','estao','voce','essa','num','suas','meu','as','minha','tem','numa','pelos','elas','havia',
              'seja','qual','sera','nos','tenho','lhe','deles','essas','esses','pelas','este','fosse','dele','tu','te','voces','vos','lhes','meus','minhas','teu','tua','teus'
              ]               

In [23]:
stop_words = stop_words1 + stop_words2
stop_words = list(set(stop_words))
stop_words = [x.lower() for x in stop_words]

In [24]:
len(stop_words)

4439

In [25]:
regex = r"(?u)\b[^\W\d]{3,}\b" 

In [26]:
df_speech = df_speech.merge(df_dep_info[['id', 'nome', 'nomeCivil']].rename(columns = {"id":"id_deputado"}), how='left', on = "id_deputado")

In [128]:
df_speech

,id_deputado,dataHoraInicio,dataHoraFim,uriEvento,faseEvento_titulo,faseEvento_dataHoraInicio,faseEvento_dataHoraFim,tipoDiscurso,urlTexto,urlAudio,urlVideo,keywords,sumario,transcricao,nome,nomeCivil
0,1595,1900-01-01 15:20:00,NaN,NaN,Ordem do Dia,NaN,NaN,DISCUSSÃO,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"TRÂNSITO, MULTA DE TRÂNSITO, RADAR ELETRÔNICO,...",Congratulações à Presidência pela condução dos...,O SR. MILTON VIEIRA (PRB - SP. Para discutir.)...,MILTON,Aristides Augusto Milton
1,1872,1900-01-01 11:01:00,NaN,NaN,Ordem do Dia,NaN,NaN,ORIENTAÇÃO DE BANCADA,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"ORIENTAÇÃO DE BANCADA, VETO 38/2017, VETO TOTA...",NaN,O SR. PEDRO CUNHA LIMA (PSDB - PB. Para encami...,CUNHA LIMA,Antonio José Maria da Cunha Lima
2,1872,1900-01-01 15:30:00,NaN,NaN,Breves Comunicações,NaN,NaN,BREVES COMUNICAÇÕES,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"RÔMULO GOUVEIA, DEPUTADO FEDERAL, FALECIMENTO,...",NaN,O SR. PEDRO CUNHA LIMA (PSDB - PB. Para uma br...,CUNHA LIMA,Antonio José Maria da Cunha Lima
3,4440,1900-01-01 11:00:00,NaN,NaN,Ordem do Dia,NaN,NaN,ENCAMINHAMENTO DE VOTAÇÃO,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"PROJETO DE LEI, CRÉDITO ESPECIAL, ALOCAÇÃO, AP...",Encaminhamento da votação do Projeto de Lei 4...,O SR. JOSÉ CARLOS ALELUIA (Bloco/PFL – BA. Par...,JOSÉ CARLOS,José Carlos de Carvalho
4,4440,1900-01-01 11:00:00,NaN,NaN,Ordem do Dia,NaN,NaN,DISCUSSÃO,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"PROJETO DE LEI, CRÉDITO ESPECIAL, ENCARGOS FIN...","Discussão do Projeto de Lei 28, de 2001-CN (Ab...",O SR. JOSÉ CARLOS ALELUIA (Bloco/PFL – BA. Par...,JOSÉ CARLOS,José Carlos de Carvalho
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454857,220558,2023-03-23 10:40:00,NaN,NaN,Breves Comunicações,NaN,NaN,BREVES COMUNICAÇÕES,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"Ensino médio,reforma\r\nLinguagem inclusiva de...",Contrariedade à implantação de linguagem neutr...,O SR. ZÉ TROVÃO (PL - SC. Sem revisão do orado...,Zé Trovão,MARCOS ANTONIO PEREIRA GOMES
454858,220558,2023-03-23 11:12:00,NaN,NaN,Breves Comunicações,NaN,NaN,PELA ORDEM,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"Trabalho por aplicativo,Regulamentação profiss...","Contrariedade à pretendida regulamentação, pel...",O SR. ZÉ TROVÃO (PL - SC. Pela ordem. Sem revi...,Zé Trovão,MARCOS ANTONIO PEREIRA GOMES
454859,220558,2023-03-27 18:36:00,NaN,NaN,Breves Comunicações,NaN,NaN,BREVES COMUNICAÇÕES,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"Governo federal,Economia,Joinville (SC),Recurs...",Gestão desastrosa do Governo Luiz Inácio Lula ...,O SR. ZÉ TROVÃO (PL - SC. Sem revisão do orado...,Zé Trovão,MARCOS ANTONIO PEREIRA GOMES
454860,220558,2023-03-28 14:12:00,NaN,NaN,Breves Comunicações,NaN,NaN,BREVES COMUNICAÇÕES,https://imagem.camara.gov.br/dc_20b.asp?largur...,NaN,NaN,"Agronegócio,Fundação Instituto de Pesquisa Eco...",NaN,O SR. ZÉ TROVÃO (PL - SC. Sem revisão do orado...,Zé Trovão,MARCOS ANTONIO PEREIRA GOMES


In [27]:
# a = df_speech.merge(df_dep_info[['id', 'nome', 'nomeCivil']].rename(columns = {"id":"id_deputado"}), how='left', on = "id_deputado")
# a = a[a.dataHoraInicio.duplicated(keep=False)][['dataHoraInicio', 'id_deputado', 'nome', 'nomeCivil', 'transcricao']]
# a['name_aux'] = a['transcricao'].str.split(' - ').str[0]
# a['name_aux'] = a['name_aux'].str.split(' \(').str[0].replace('O SR. ','', regex=True).replace('O SRA. ','', regex=True)
# a[['id_deputado', 'nome', 'nomeCivil','name_aux']].drop_duplicates().to_excel(os.path.join(interim_path, 'speech_aux.xlsx'), index=False)

In [28]:
# a[a.dataHoraInicio>'1901-01-01'].sort_values('dataHoraInicio')
# [['id_deputado', 'nome', 'nomeCivil', 'transcricao']].drop_duplicates()

## TF-IDF 

**TF-IDF - Term Frequency-Inverse Document Frequency.**:
 - It is a statistical measure used to evaluate the importance of a word in a document relative to a collection of documents (corpus). The TF-IDF value increases proportionally to the number of times a word appears in the document but is offset by the frequency of the word in the corpus, which helps to adjust for the fact that some words appear more frequently in general.
    
**Components:**

- **Term Frequency (TF):** Measures how frequently a term occurs in a document. The more a term appears in a document, the higher its TF score.

   - $\text{TF}(t) = \frac{\text{Number of times term } t \text{ appears in a document}}{\text{Total number of terms in the document}}$
  
- **Inverse Document Frequency (IDF):** Measures how important a term is. While computing TF, all terms are considered equally important. However, some terms like "is", "of", and "that" may appear a lot but have little importance. Thus, we need to weigh down the frequent terms while scaling up the rare ones.

   - $\text{IDF}(t) = \log\left(\frac{\text{Total number of documents}}{\text{Number of documents with term } t \text{ in it}}\right)$

- **TF-IDF:** The TF-IDF score for a term is the product of its TF and IDF scores.

   - $\text{TF-IDF}(t) = \text{TF}(t) \times \text{IDF}(t)$

**How `TfidfVectorizer` Works:**

1. **Tokenization:** Splits the text into words (tokens).
2. **Building the Vocabulary:** Creates a dictionary of all unique words across the corpus.
3. **Calculating TF-IDF Scores:** Computes the TF-IDF score for each word in each document.

**Parameters Used in Your Code:**

- `min_df=10`: Ignore terms that appear in fewer than 10 documents.
- `max_df=0.95`: Ignore terms that appear in more than 95% of the documents.
- `stop_words=stop_words`: A list of words to ignore (commonly used words that are not informative).
- `token_pattern=regex`: A regex pattern to tokenize the text.

**Application to Deputies' Speeches:**

By applying `TfidfVectorizer` to deputies' speeches, you are converting their speeches into numerical features that can be used for various machine learning tasks, such as:

- **Text Classification:** Classify speeches by topics, sentiments, or authors.
- **Clustering:** Group similar speeches together.
- **Information Retrieval:** Find speeches similar to a query speech.
- **Feature Extraction:** Extract meaningful features for further analysis.



In [37]:
id_cols = ['id_deputado', 'nome', 'dataHoraInicio', 'nomeCivil']
year_list = df_speech.dataHoraInicio.dt.year.unique()

In [38]:
dict_tfidf = {x:{'words':[],'data':[]} for x in dict_tematic}

In [39]:
df_final = pd.DataFrame()
for year in sorted(year_list[1:]):
    print(year)
    df_speech_f = df_speech[df_speech.dataHoraInicio.dt.year == year]
    # df_speech_f = df_speech_f.merge(df_dep_info[['id', 'nome', 'nomeCivil']].rename(columns = {"id":"id_deputado"}), how='left', on = "id_deputado")
    df_speech_f = df_speech_f.set_index(['id_deputado', 'nome', 'dataHoraInicio'])
    
    vectorizer = TfidfVectorizer(min_df= 10,  max_df = 0.95, stop_words = stop_words, token_pattern = regex)
    
    X = vectorizer.fit_transform(df_speech_f['transcricao'])
    y = df_speech_f['nomeCivil']
    
    features = vectorizer.get_feature_names_out()
    
    tfidf_dense = X.todense()
    tfidf_df = pd.DataFrame(tfidf_dense, columns=features)
    tfidf_df = pd.concat([y.reset_index(), tfidf_df],axis=1)
    
    df_id = tfidf_df[id_cols]    
    for k,v in dict_tematic.items():
        filtered_list = []

        values = [word for phrase in v for word in phrase.split()]

        for phrase in values:
            words = phrase.split()
            if len(phrase) > 1:
                if all(word in features for word in words):
                    filtered_list.extend(words)
            else:
                filtered_list.extend([item for item in features if words[0] in item and item not in filtered_list])

        filtered_list = list(dict.fromkeys(filtered_list))

        df_id = pd.concat([df_id,tfidf_df[filtered_list].mean(axis=1)],axis=1).rename(columns = {0:k})
        # print('theme:',k)
        # print(' - words related:',v)
        # print(' - words in the speech:',filtered_list) 
    
    df_id['year'] = year
    df_final = pd.concat([df_final, df_id])
    

2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023


In [130]:
tfidf_df.head()

,id_deputado,nome,dataHoraInicio,nomeCivil,abaixo,abandonadas,abandonados,abandono,abastecimento,abc,...,órfãos,órgão,órgãos,ônibus,ônus,última,últimas,últimos,única,único
0,73701,Benedita da Silva,2023-02-08 13:56:00,BENEDITA SOUZA DA SILVA SAMPAIO,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.07454
1,73701,Benedita da Silva,2023-02-09 09:24:00,BENEDITA SOUZA DA SILVA SAMPAIO,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
2,73701,Benedita da Silva,2023-02-28 14:08:00,BENEDITA SOUZA DA SILVA SAMPAIO,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
3,73701,Benedita da Silva,2023-03-01 14:52:00,BENEDITA SOUZA DA SILVA SAMPAIO,0.0,0.0,0.076904,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
4,73701,Benedita da Silva,2023-03-02 10:12:00,BENEDITA SOUZA DA SILVA SAMPAIO,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000


In [106]:
df_final

,id_deputado,nome,nome,dataHoraInicio,nomeCivil,Agropecuária,Cidades e Transportes,"Ciência, Tecnologia e comunicações",Economia & Consumidor,Direitos Humanos,"Educação, cultura e esportes",Meio ambiente e energia,Política e administração pública,Relações exteriores,Saúde,Segurança,"Trabalho, previdência e assistência",year
0,73468,ANTÔNIO CARLOS KONDER REIS,0.000000,2000-10-05 14:02:00,ANTÔNIO CARLOS KONDER REIS,0.000000,0.000000,0.000000,0.002691,0.0,0.002244,0.000000,0.015380,0.004310,0.000000,0.000000,0.006255,2000
1,73468,ANTÔNIO CARLOS KONDER REIS,0.040821,2000-10-05 16:18:00,ANTÔNIO CARLOS KONDER REIS,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.016305,2000
2,73468,ANTÔNIO CARLOS KONDER REIS,0.000000,2000-10-17 14:58:00,ANTÔNIO CARLOS KONDER REIS,0.000000,0.000000,0.012182,0.000000,0.0,0.000000,0.000000,0.026715,0.004572,0.000000,0.000000,0.000000,2000
3,73468,ANTÔNIO CARLOS KONDER REIS,0.015619,2000-11-06 17:26:00,ANTÔNIO CARLOS KONDER REIS,0.000000,0.004488,0.004725,0.006387,0.0,0.001107,0.002338,0.013279,0.007233,0.003765,0.000000,0.000000,2000
4,73468,ANTÔNIO CARLOS KONDER REIS,0.071640,2000-11-08 10:16:00,ANTÔNIO CARLOS KONDER REIS,0.002898,0.012431,0.000000,0.005517,0.0,0.003426,0.000000,0.042059,0.009778,0.000000,0.032162,0.008554,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2571,220558,Zé Trovão,0.000000,2023-03-23 10:40:00,MARCOS ANTONIO PEREIRA GOMES,0.014285,0.000000,0.000000,0.000000,0.0,0.013927,0.000000,0.000000,0.000000,0.007414,0.000000,0.000000,2023
2572,220558,Zé Trovão,0.000000,2023-03-23 11:12:00,MARCOS ANTONIO PEREIRA GOMES,0.000000,0.000000,0.000000,0.037172,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2023
2573,220558,Zé Trovão,0.000000,2023-03-27 18:36:00,MARCOS ANTONIO PEREIRA GOMES,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.007862,0.051926,0.004399,0.007166,0.000000,0.000000,2023
2574,220558,Zé Trovão,0.000000,2023-03-28 14:12:00,MARCOS ANTONIO PEREIRA GOMES,0.047561,0.000000,0.000000,0.004565,0.0,0.013172,0.000000,0.000000,0.011479,0.000000,0.014910,0.000000,2023


## Translate speechs

In [ ]:
translator = Translator()

In [ ]:
i=0
translation_list = []
speechs = df_speech['transcricao'].to_list()

In [155]:
for txt in speechs:    
    
    if i%5000 == 0:
        print(i,len(txt))
    try:
        translation = translate_text(txt)        
    except:
        time.sleep(1)
        try:
            translation = translate_text(txt)
        except:
            time.sleep(5)
            try:
                translation = translate_text(txt)
            except:
                translation = np.nan
                time.sleep(5)
                
    df_tmp = pd.DataFrame({'transcricao':txt,'traducao':translation},index=[i])        
    translation_list.append(df_tmp)
     
    if i%50000 == 0:        
        df_speechs_aux = pd.concat(translation_list)
        df_speechs_aux.to_csv(os.path.join(interim_path, 'translation_speechs_.csv'),index=False)   
    
    i += 1

405000 522
406000 205
407000 3054
408000 1416
409000 471
410000 17915
411000 379
412000 2902
413000 863
414000 3175
415000 1391
416000 1870
417000 3899
418000 785
419000 4372
420000 5031
421000 2997
422000 731
423000 2814
424000 221
425000 2534
426000 916
427000 3439
428000 3540
429000 1333
430000 1206
431000 3204
432000 1206
433000 2018
434000 566
435000 1766
436000 2774
437000 2261
438000 4943
439000 3161
440000 290
441000 7025
442000 2709
443000 1151
444000 4811
445000 8983
446000 1306
447000 598
448000 5063
449000 1546
450000 402
451000 255
452000 2032
453000 1658
454000 905
455000 3999
456000 3328
457000 2382
458000 1201
459000 1055
460000 2600
461000 577
462000 1924
463000 1410
464000 1481
465000 2037
466000 2484
467000 5733
468000 5785
469000 4749
470000 472
471000 745
472000 274
473000 3144
474000 3984
475000 1899
476000 839
477000 2705
478000 577
479000 5604
480000 780
481000 2468
482000 990
483000 5371
484000 596
485000 2670
486000 583
487000 5160
488000 1388
489000 1824
4900

In [196]:
len(translation_list)

454862

In [259]:
df_translation = pd.concat(translation_list)
df_translation.to_csv(os.path.join(interim_path, 'translation_speechs_raw.csv'),index=False, sep = ';')   

In [200]:
df_speech_ = df_speech[['id_deputado', 'dataHoraInicio', 'faseEvento_titulo','tipoDiscurso', 'keywords', 'sumario', 'transcricao']]
df_speech_ =  df_speech_.merge(df_translation.reset_index(drop = True).drop('transcricao',axis=1), left_index=True, right_index=True)

In [ ]:
speechs_missing = df_speech_[df_speech_.traducao.isna()].transcricao.to_list()

In [233]:
translation_list2 = []
for i,txt in enumerate(speechs_missing):
    print(i)
    translation = translate_text(txt)        
    df_tmp = pd.DataFrame({'transcricao':txt,'traducao':translation},index=[i])        
    translation_list2.append(df_tmp)

In [257]:
df_translation2 = pd.concat(translation_list2)

In [255]:
df_speech_ = df_speech_.merge(df_translation2.rename(columns = {'traducao':'traducao_aux'}),how='left', on=['transcricao'])
df_speech_.loc[df_speech_['traducao'].isna(), 'traducao'] = df_speech_['traducao_aux']
df_speech_ = df_speech_.drop('traducao_aux',axis=1)

In [260]:
df_speech_.to_csv(os.path.join(interim_path, 'translation_speechs.csv'),index=False, sep = ';')   

## Sentiment Analyzer

1. **VADER (Valence Aware Dictionary and sEntiment Reasoner) nuanced analysis**:
    - The SentimentIntensityAnalyzer is designed to capture subtle variations in sentiment, which is why it provides separate scores for negative, neutral, and positive sentiments. The compound score gives an overall sentiment assessment. In this case, it recognized both positive and neutral aspects of the sentence but determined the overall sentiment as positive.
       - **neg**: Proportion of the text that is negative (0.0 means no negativity detected).
       - **neu**: Proportion of the text that is neutral (0.651 means the majority of the text is neutral).
       - **pos**: Proportion of the text that is positive (0.349 means a significant portion is positive).
       - **compound**: An aggregated score that summarizes the overall sentiment. It ranges from -1 (most negative) to +1 (most positive). A score of 0.6468 indicates a positive sentiment.

2. **TextBlob Results:**:
   - TextBlob simplifies the sentiment into two scores: polarity and subjectivity. For your text, TextBlob interpreted the sentiment as maximally positive (polarity=1.0) and highly subjective (subjectivity=1.0). TextBlob doesn’t break down sentiment into components like VADER does but gives a straightforward positive or negative assessment.
        - **polarity**: This score ranges from -1 (very negative) to +1 (very positive). A polarity of 1.0 indicates the text is extremely positive.
        - **subjectivity**: This score ranges from 0 (very objective) to 1 (very subjective). A subjectivity of 1.0 indicates that the text is entirely based on personal opinions or feelings.



In [72]:
sia = SentimentIntensityAnalyzer()

In [71]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Pichau\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [82]:
df_speech_transl = pd.read_csv(os.path.join(interim_path, 'translation_speechs.csv'), sep = ';')   

In [100]:
df_speech_transl['neg'], df_speech_transl['neu'], df_speech_transl['pos'], df_speech_transl['compound'] = zip(*df_speech_transl['traducao'].apply(lambda x: get_sentiment(x, 'sia')))

In [102]:
df_speech_transl['polarity'], df_speech_transl['subjectivity'] = zip(*df_speech_transl['traducao'].apply(lambda x: get_sentiment(x, 'blob')))

In [103]:
df_speech_transl

,id_deputado,dataHoraInicio,faseEvento_titulo,tipoDiscurso,keywords,sumario,transcricao,traducao,neg,neu,pos,compound,polarity,subjectivity
0,1595,1900-01-01 15:20:00,Ordem do Dia,DISCUSSÃO,"TRÂNSITO, MULTA DE TRÂNSITO, RADAR ELETRÔNICO,...",Congratulações à Presidência pela condução dos...,O SR. MILTON VIEIRA (PRB - SP. Para discutir.)...,Mr.Milton Vieira (PRB - SP. To discuss.) - Mr....,0.059,0.825,0.116,0.7728,0.185833,0.580833
1,1872,1900-01-01 11:01:00,Ordem do Dia,ORIENTAÇÃO DE BANCADA,"ORIENTAÇÃO DE BANCADA, VETO 38/2017, VETO TOTA...",NaN,O SR. PEDRO CUNHA LIMA (PSDB - PB. Para encami...,Mr.Pedro Cunha Lima (PSDB - PB. To forward. Wi...,0.021,0.941,0.038,0.0953,-0.001364,0.361667
2,1872,1900-01-01 15:30:00,Breves Comunicações,BREVES COMUNICAÇÕES,"RÔMULO GOUVEIA, DEPUTADO FEDERAL, FALECIMENTO,...",NaN,O SR. PEDRO CUNHA LIMA (PSDB - PB. Para uma br...,Mr.Pedro Cunha Lima (PSDB - PB. For a brief co...,0.057,0.679,0.265,0.9989,0.370000,0.521772
3,4440,1900-01-01 11:00:00,Ordem do Dia,ENCAMINHAMENTO DE VOTAÇÃO,"PROJETO DE LEI, CRÉDITO ESPECIAL, ALOCAÇÃO, AP...",Encaminhamento da votação do Projeto de Lei 4...,O SR. JOSÉ CARLOS ALELUIA (Bloco/PFL – BA. Par...,Mr.José Carlos Aleluia (Bloco/PFL - BA. To for...,0.039,0.868,0.093,0.8841,0.234935,0.541688
4,4440,1900-01-01 11:00:00,Ordem do Dia,DISCUSSÃO,"PROJETO DE LEI, CRÉDITO ESPECIAL, ENCARGOS FIN...","Discussão do Projeto de Lei 28, de 2001-CN (Ab...",O SR. JOSÉ CARLOS ALELUIA (Bloco/PFL – BA. Par...,Mr.José Carlos Aleluia (Block/PFL - BA. To dis...,0.018,0.910,0.072,0.8126,0.071429,0.659524
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454857,220558,2023-03-23 10:40:00,Breves Comunicações,BREVES COMUNICAÇÕES,"Ensino médio,reforma\r\nLinguagem inclusiva de...",Contrariedade à implantação de linguagem neutr...,O SR. ZÉ TROVÃO (PL - SC. Sem revisão do orado...,Mr.Zé Trovão (PL - SC. Without speaker review....,0.105,0.809,0.087,-0.7434,0.037290,0.586468
454858,220558,2023-03-23 11:12:00,Breves Comunicações,PELA ORDEM,"Trabalho por aplicativo,Regulamentação profiss...","Contrariedade à pretendida regulamentação, pel...",O SR. ZÉ TROVÃO (PL - SC. Pela ordem. Sem revi...,Mr.Zé Trovão (PL - SC. By order. Without revie...,0.073,0.812,0.116,0.6291,0.037778,0.552778
454859,220558,2023-03-27 18:36:00,Breves Comunicações,BREVES COMUNICAÇÕES,"Governo federal,Economia,Joinville (SC),Recurs...",Gestão desastrosa do Governo Luiz Inácio Lula ...,O SR. ZÉ TROVÃO (PL - SC. Sem revisão do orado...,Mr.Zé Trovão (PL - SC. Without review of the s...,0.078,0.805,0.117,0.9578,0.218933,0.574113
454860,220558,2023-03-28 14:12:00,Breves Comunicações,BREVES COMUNICAÇÕES,"Agronegócio,Fundação Instituto de Pesquisa Eco...",NaN,O SR. ZÉ TROVÃO (PL - SC. Sem revisão do orado...,Mr.Zé Trovão (PL - SC. Without speaker review....,0.113,0.782,0.104,-0.7324,-0.034947,0.562264


In [112]:
df_speech_transl[-3:-2].transcricao.item()

'O SR. ZÉ TROVÃO (PL - SC. Sem revisão do orador.) - Sr. Presidente, quero inicialmente saudá-lo e dizer que é sempre uma honra ver a sessão ser presidida por V.Exa., um homem que tem responsabilidade, que tem bom ânimo e que conduz os trabalhos sempre com muita grandeza. Agradeço aos nobres pares da Direita ou da conhecida pela Esquerda como extrema-direita. Na verdade, nós não estamos na extrema-direita, nós somos a extrema necessidade para um país que corre grandes riscos na mão do tal "bandidim de nove dedos" \x97 é assim que eu vou chamá-lo; vou usar até um pronome meio neutro, para depois não ser processado \x97, que só faz o Brasil andar para trás. Noventa e oito mil: este é o número dos pontos, é a nossa pontuação que caiu na Bolsa de Valores. Isso reflete os 20% que o Brasil está regredindo todos os dias. Mas alguém vai à outra tribuna encher a boca para dizer que o descondenado, mais conhecido como "bandido", está fazendo política séria. Nunca vi este desgoverno fazer polític

In [113]:
df_speech_transl[-3:-2].traducao.item()

'Mr.Zé Trovão (PL - SC. Without review of the speaker.) - Mr. President, I want initially healthy and say that it is always an honor to see the session to be presided by you., A man who has responsibility, who has goodCEO and that always conducts work with great greatness.I thank the noble pairs of the right or the known to the left as far right.In fact, we are not in the far right, we are the extreme need for a country that runs great risks in the hand of such "nine-fingers" that\'s how I will call it;I will even use a kind of neutral pronoun, so that it is not processed, which only makes Brazil walk backwards.Ninety -eight thousand: This is the number of points, it is our score that fell on the stock exchange.This reflects the 20% that Brazil is regressing every day.But someone goes to the other tribune to fill his mouth to say that the discount, better known as "bandit", is doing serious politics.I have never seen this mischief do serious politics.This mischief is not ashamed in the

In [114]:
txt = df_speech_transl[-3:-2].traducao.item()
len(txt)

2572

In [117]:
sia.polarity_scores(txt) 

{'neg': 0.078, 'neu': 0.805, 'pos': 0.117, 'compound': 0.9578}

In [119]:
TextBlob(txt).sentiment

Sentiment(polarity=0.21893267651888343, subjectivity=0.5741133004926108)

In [132]:
df_speech_transl.iloc[-3]

id_deputado                                                     220558
dataHoraInicio                                     2023-03-27 18:36:00
faseEvento_titulo                                  Breves Comunicações
tipoDiscurso                                       BREVES COMUNICAÇÕES
keywords             Governo federal,Economia,Joinville (SC),Recurs...
sumario              Gestão desastrosa do Governo Luiz Inácio Lula ...
transcricao          O SR. ZÉ TROVÃO (PL - SC. Sem revisão do orado...
traducao             Mr.Zé Trovão (PL - SC. Without review of the s...
neg                                                              0.078
neu                                                              0.805
pos                                                              0.117
compound                                                        0.9578
polarity                                                      0.218933
subjectivity                                                  0.574113
Name: 

In [131]:
df_final.iloc[-3]

id_deputado                                                  220558
nome                                                      Zé Trovão
nome                                                            0.0
dataHoraInicio                                  2023-03-27 18:36:00
nomeCivil                              MARCOS ANTONIO PEREIRA GOMES
Agropecuária                                                    0.0
Cidades e Transportes                                           0.0
Ciência, Tecnologia e comunicações                              0.0
Economia & Consumidor                                           0.0
Direitos Humanos                                                0.0
Educação, cultura e esportes                                    0.0
Meio ambiente e energia                                    0.007862
Política e administração pública                           0.051926
Relações exteriores                                        0.004399
Saúde                                           